# Install Dependencies

This cell installs the required packages for YOLOv8 training and YAML parsing.

In [ ]:
!pip install ultralytics pyyaml

# Configuration

Centralized model and dataset configuration values used throughout the notebook.


In [ ]:
MODEL_NAME = "yolov8n.pt"
DATASET_YAML = "../datasets/data.yaml"
IMAGE_SIZE = 640
EPOCHS = 100
BATCH_SIZE = 16
OUTPUT_DIR = Path("../models").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Verify Dataset

Confirm the fixed YOLOv8 dataset structure at `ai/datasets/` before training.


In [ ]:
from pathlib import Path
import yaml

dataset_root = Path(DATASET_YAML).resolve().parent
data_yaml = Path(DATASET_YAML).resolve()

assert dataset_root.exists(), f'Dataset root {dataset_root} does not exist'
assert data_yaml.exists(), f'{DATASET_YAML} not found'

with open(data_yaml, 'r') as f:
    cfg = yaml.safe_load(f)

required_dirs = [
    dataset_root / 'train' / 'images',
    dataset_root / 'train' / 'labels',
    dataset_root / 'valid' / 'images',
    dataset_root / 'valid' / 'labels',
    dataset_root / 'test' / 'images',
    dataset_root / 'test' / 'labels',
]
for path in required_dirs:
    assert path.exists(), f'Required path not found: {path}'

train_count = sum(1 for _ in (dataset_root / 'train' / 'images').glob('*.jpg')) + sum(1 for _ in (dataset_root / 'train' / 'images').glob('*.jpeg')) + sum(1 for _ in (dataset_root / 'train' / 'images').glob('*.png'))
valid_count = sum(1 for _ in (dataset_root / 'valid' / 'images').glob('*.jpg')) + sum(1 for _ in (dataset_root / 'valid' / 'images').glob('*.jpeg')) + sum(1 for _ in (dataset_root / 'valid' / 'images').glob('*.png'))
test_count = sum(1 for _ in (dataset_root / 'test' / 'images').glob('*.jpg')) + sum(1 for _ in (dataset_root / 'test' / 'images').glob('*.jpeg')) + sum(1 for _ in (dataset_root / 'test' / 'images').glob('*.png'))

print('data.yaml exists:', data_yaml.exists())
print('train images:', train_count)
print('valid images:', valid_count)
print('test images:', test_count)
print('class names:', cfg.get('names') or cfg.get('class_names') or cfg.get('labels'))


# Train YOLOv8

Train YOLOv8 Nano with the fixed dataset and specified training configuration.


In [ ]:
from pathlib import Path
import torch
from ultralytics import YOLO

dataset_yaml = Path(DATASET_YAML).resolve()

device = 'mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

model = YOLO(MODEL_NAME)
model.train(
    data=str(dataset_yaml),
    imgsz=IMAGE_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    device=device,
    project=str(OUTPUT_DIR),
    name='train',
    exist_ok=True,
)
print('Training completed.')


# Validate Model

Run model validation on the trained weights to confirm the result.


In [ ]:
from pathlib import Path
from ultralytics import YOLO

weights_path = OUTPUT_DIR / 'train' / 'weights' / 'best.pt'
assert weights_path.exists(), f'Expected trained best.pt not found at {weights_path}'

model = YOLO(str(weights_path))
results = model.val(data=str(DATASET_YAML), imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device='cpu')
print('Validation completed.')
print('Validation results:', results.metrics if hasattr(results, 'metrics') else results)


# Export best.pt

Copy the final `best.pt` file into `ai/models/best.pt` for standardized access.


In [ ]:
from pathlib import Path
import shutil

weights_path = (OUTPUT_DIR / 'train' / 'weights' / 'best.pt').resolve()
assert weights_path.exists(), f'best.pt not found at {weights_path}'

best_destination = (OUTPUT_DIR / 'best.pt').resolve()
best_destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(weights_path, best_destination)

print('✓ best.pt exists')
print('✓ path', best_destination)
print('✓ model size (MB)', round(best_destination.stat().st_size / (1024 * 1024), 2))
